# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
%pip install -q tensorflow numpy matplotlib

import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

if not hasattr(tf.keras.metrics.Metric, "reset_states"):
    tf.keras.metrics.Metric.reset_states = tf.keras.metrics.Metric.reset_state

device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print_every = 100

%matplotlib inline


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
        )
        self.conv2 = tf.keras.layers.Conv2D(
            channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer,
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 4 and x.shape[1] == 3 and x.shape[-1] != 3:
            x = tf.transpose(x, perm=[0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores


In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_states()
            train_accuracy.reset_states()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_states()
                        val_accuracy.reset_states()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.9608559608459473, Accuracy: 9.375, Val Loss: 2.8383452892303467, Val Accuracy: 14.200000762939453
Iteration 100, Epoch 1, Loss: 2.2318060398101807, Accuracy: 28.341585159301758, Val Loss: 1.8909177780151367, Val Accuracy: 36.39999771118164
Iteration 200, Epoch 1, Loss: 2.073538303375244, Accuracy: 32.36940383911133, Val Loss: 1.909645676612854, Val Accuracy: 37.79999923706055
Iteration 300, Epoch 1, Loss: 1.9974217414855957, Accuracy: 34.19331359863281, Val Loss: 1.8700697422027588, Val Accuracy: 37.099998474121094
Iteration 400, Epoch 1, Loss: 1.9272409677505493, Accuracy: 36.01543045043945, Val Loss: 1.7260593175888062, Val Accuracy: 41.0
Iteration 500, Epoch 1, Loss: 1.8820347785949707, Accuracy: 37.197479248046875, Val Loss: 1.6660832166671753, Val Accuracy: 43.099998474121094
Iteration 600, Epoch 1, Loss: 1.8514961004257202, Accuracy: 38.121360778808594, Val Loss: 1.6924222707748413, Val Accuracy: 43.0
Iteration 700, Epoch 1, Loss: 1.8258212804794312,

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True,
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.567002773284912, Accuracy: 10.9375, Val Loss: 2.74981951713562, Val Accuracy: 11.300000190734863
Iteration 100, Epoch 1, Loss: 1.8137401342391968, Accuracy: 35.689971923828125, Val Loss: 1.6275914907455444, Val Accuracy: 44.70000076293945
Iteration 200, Epoch 1, Loss: 1.6820138692855835, Accuracy: 40.78047180175781, Val Loss: 1.463894009590149, Val Accuracy: 48.0
Iteration 300, Epoch 1, Loss: 1.603538155555725, Accuracy: 43.433345794677734, Val Loss: 1.4221140146255493, Val Accuracy: 50.400001525878906
Iteration 400, Epoch 1, Loss: 1.5320818424224854, Accuracy: 45.889183044433594, Val Loss: 1.3444840908050537, Val Accuracy: 52.60000228881836
Iteration 500, Epoch 1, Loss: 1.4830976724624634, Accuracy: 47.63285827636719, Val Loss: 1.284562587738037, Val Accuracy: 56.099998474121094
Iteration 600, Epoch 1, Loss: 1.4514764547348022, Accuracy: 48.733882904052734, Val Loss: 1.2395515441894531, Val Accuracy: 57.79999923706055
Iteration 700, Epoch 1, Loss: 1.42134

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.231055736541748, Accuracy: 4.6875, Val Loss: 3.095139503479004, Val Accuracy: 11.59999942779541
Iteration 100, Epoch 1, Loss: 2.262934684753418, Accuracy: 28.27970314025879, Val Loss: 1.90628981590271, Val Accuracy: 35.79999923706055
Iteration 200, Epoch 1, Loss: 2.0897085666656494, Accuracy: 32.136192321777344, Val Loss: 1.8376868963241577, Val Accuracy: 39.89999771118164
Iteration 300, Epoch 1, Loss: 2.0077738761901855, Accuracy: 34.08430099487305, Val Loss: 1.8846954107284546, Val Accuracy: 37.20000076293945
Iteration 400, Epoch 1, Loss: 1.9377143383026123, Accuracy: 35.89463806152344, Val Loss: 1.7127755880355835, Val Accuracy: 41.0
Iteration 500, Epoch 1, Loss: 1.8934006690979004, Accuracy: 36.988525390625, Val Loss: 1.6514344215393066, Val Accuracy: 43.39999771118164
Iteration 600, Epoch 1, Loss: 1.8620617389678955, Accuracy: 37.85877990722656, Val Loss: 1.6863353252410889, Val Accuracy: 41.0
Iteration 700, Epoch 1, Loss: 1.8354246616363525, Accuracy

Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.8139 - sparse_categorical_accuracy: 0.3926 - val_loss: 1.5873 - val_sparse_categorical_accuracy: 0.4710
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.6130 - sparse_categorical_accuracy: 0.4438


[1.6130282878875732, 0.4438000023365021]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            32,
            kernel_size=5,
            padding='same',
            activation='relu',
            input_shape=(32, 32, 3),
            kernel_initializer=initializer,
        ),
        tf.keras.layers.Conv2D(
            16,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer),
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True,
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.9306323528289795, Accuracy: 4.6875, Val Loss: 2.5298516750335693, Val Accuracy: 10.5
Iteration 100, Epoch 1, Loss: 2.0092737674713135, Accuracy: 29.238861083984375, Val Loss: 1.7983758449554443, Val Accuracy: 37.900001525878906
Iteration 200, Epoch 1, Loss: 1.8807110786437988, Accuracy: 33.93967819213867, Val Loss: 1.6663901805877686, Val Accuracy: 42.5
Iteration 300, Epoch 1, Loss: 1.8068825006484985, Accuracy: 36.736915588378906, Val Loss: 1.6168638467788696, Val Accuracy: 45.0
Iteration 400, Epoch 1, Loss: 1.7421764135360718, Accuracy: 38.86377716064453, Val Loss: 1.5346437692642212, Val Accuracy: 47.099998474121094
Iteration 500, Epoch 1, Loss: 1.6962029933929443, Accuracy: 40.51896286010742, Val Loss: 1.4883596897125244, Val Accuracy: 48.20000076293945
Iteration 600, Epoch 1, Loss: 1.6653612852096558, Accuracy: 41.586936950683594, Val Loss: 1.4584866762161255, Val Accuracy: 49.39999771118164
Iteration 700, Epoch 1, Loss: 1.6378631591796875, Accuracy: 

In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.6315 - sparse_categorical_accuracy: 0.4276 - val_loss: 1.4018 - val_sparse_categorical_accuracy: 0.5040
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.4501 - sparse_categorical_accuracy: 0.4862


[1.450063943862915, 0.4862000048160553]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0130057334899902, Accuracy: 10.9375, Val Loss: 3.0397353172302246, Val Accuracy: 13.500000953674316
Iteration 100, Epoch 1, Loss: 2.2268221378326416, Accuracy: 28.5272274017334, Val Loss: 1.8648982048034668, Val Accuracy: 38.80000305175781
Iteration 200, Epoch 1, Loss: 2.079737663269043, Accuracy: 32.299442291259766, Val Loss: 1.8235167264938354, Val Accuracy: 39.599998474121094
Iteration 300, Epoch 1, Loss: 2.0037474632263184, Accuracy: 34.1673583984375, Val Loss: 1.8589608669281006, Val Accuracy: 37.900001525878906
Iteration 400, Epoch 1, Loss: 1.9333162307739258, Accuracy: 36.00763702392578, Val Loss: 1.7375826835632324, Val Accuracy: 41.80000305175781
Iteration 500, Epoch 1, Loss: 1.8903051614761353, Accuracy: 37.100799560546875, Val Loss: 1.654421091079712, Val Accuracy: 45.0
Iteration 600, Epoch 1, Loss: 1.8605659008026123, Accuracy: 37.88737487792969, Val Loss: 1.667742133140564, Val Accuracy: 42.79999923706055
Iteration 700, Epoch 1, Loss: 1.834254

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.keras.initializers.HeNormal()
        self.conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=2)
        self.drop1 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(128, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=2)
        self.drop2 = tf.keras.layers.Dropout(0.3)

        self.conv5 = tf.keras.layers.Conv2D(256, 3, padding='same', use_bias=False, kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)
        self.drop3 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.gap(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)


Iteration 0, Epoch 1, Loss: 2.719766855239868, Accuracy: 3.125, Val Loss: 4.81602144241333, Val Accuracy: 8.699999809265137
Iteration 700, Epoch 1, Loss: 1.4421648979187012, Accuracy: 47.22494888305664, Val Loss: 1.1684110164642334, Val Accuracy: 58.70000076293945
Iteration 1400, Epoch 2, Loss: 1.0604054927825928, Accuracy: 62.22932815551758, Val Loss: 1.1449869871139526, Val Accuracy: 60.000003814697266
Iteration 2100, Epoch 3, Loss: 0.9166771173477173, Accuracy: 67.51702117919922, Val Loss: 0.9379070401191711, Val Accuracy: 69.0
Iteration 2800, Epoch 4, Loss: 0.81134432554245, Accuracy: 71.57057189941406, Val Loss: 0.9069103002548218, Val Accuracy: 69.19999694824219
Iteration 3500, Epoch 5, Loss: 0.7335164546966553, Accuracy: 74.48155212402344, Val Loss: 0.8662991523742676, Val Accuracy: 70.30000305175781
Iteration 4200, Epoch 6, Loss: 0.6663131713867188, Accuracy: 77.40060424804688, Val Loss: 0.7519717216491699, Val Accuracy: 74.69999694824219
Iteration 4900, Epoch 7, Loss: 0.605733

Опишите все эксперименты, результаты. Сделайте выводы.

1. Базовые модели (двухслойная полносвязная сеть и простая трехслойная CNN) показывают ограниченное качество на CIFAR-10 за 1 эпоху: полносвязная модель достигает `44.8%` на валидации, трехслойная CNN с Nesterov momentum (`0.9`) улучшает результат до `57.9%`, что подтверждает преимущество сверточной архитектуры для изображений.
2. Реализация через `tf.keras.Sequential` для трехслойной CNN дает сопоставимую, но чуть более низкую точность (около `49.6%`), что показывает чувствительность результата к выбору гиперпараметров (в частности, learning rate и настройкам оптимизации), а не только к API реализации.
3. Улучшенная архитектура `CustomConvNet` с Batch Normalization, Dropout и более глубокой сверточной частью за 10 эпох существенно повышает качество и достигает `82.7%` на валидационной выборке.
4. Целевое требование задания (`не менее 70%` за 10 эпох) уверенно выполнено. Использование нормализации, регуляризации и более выразительной архитектуры значительно повышает обобщающую способность модели и устойчивость обучения.